# SHARP CEA Cartesian Extrapolation

Minimal notebook using the NF2 Python API. Edit the paths, active-region number, and time; then run the cells.

In [ ]:
from pathlib import Path

RUN_DOWNLOAD = True
RUN_TRAINING = True

JSOC_EMAIL = "you@example.org"
SHARP_NUM = 377
NOAA_NUM = None
T_START = "2011-02-15T00:00:00"
T_END = "2011-02-15T00:12:00"
CADENCE = "720s"
SERIES = "sharp_cea_720s"
SEGMENTS = "Br,Bp,Bt,Br_err,Bp_err,Bt_err"

RUN_DIR = Path("runs/sharp_cea")
DATA_DIR = RUN_DIR / "data"
WORK_DIR = RUN_DIR / "work"
NF2_PATH = RUN_DIR / "extrapolation_result.nf2"
EXPORT_DIR = RUN_DIR / "exports"

BR_FITS = DATA_DIR / "Br.fits"
BT_FITS = DATA_DIR / "Bt.fits"
BP_FITS = DATA_DIR / "Bp.fits"
BR_ERR_FITS = DATA_DIR / "Br_err.fits"
BT_ERR_FITS = DATA_DIR / "Bt_err.fits"
BP_ERR_FITS = DATA_DIR / "Bp_err.fits"

EXPORT_MM_PER_PIXEL = 1.44
HEIGHT_MIN = 0
HEIGHT_MAX = 80

In [ ]:
from pathlib import Path
from dateutil.parser import parse

import matplotlib.pyplot as plt
import numpy as np

import nf2
from nf2.evaluation.metric import normalized_divergence, sigma_J, theta_J, vector_norm
from nf2.evaluation.output_metrics import current_density

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
if RUN_DOWNLOAD:
    nf2.download_sharp_series(
        str(DATA_DIR), JSOC_EMAIL, parse(T_START), parse(T_END),
        noaa_num=NOAA_NUM, sharp_num=SHARP_NUM, cadence=CADENCE,
        segments=SEGMENTS, series=SERIES,
    )

def find_segment(segment, fallback):
    matches = sorted(DATA_DIR.glob(f"*.{segment}.fits"))
    return matches[0] if matches else fallback

if not BR_FITS.exists():
    BR_FITS = find_segment("Br", BR_FITS)
if not BT_FITS.exists():
    BT_FITS = find_segment("Bt", BT_FITS)
if not BP_FITS.exists():
    BP_FITS = find_segment("Bp", BP_FITS)
if not BR_ERR_FITS.exists():
    BR_ERR_FITS = find_segment("Br_err", BR_ERR_FITS)
if not BT_ERR_FITS.exists():
    BT_ERR_FITS = find_segment("Bt_err", BT_ERR_FITS)
if not BP_ERR_FITS.exists():
    BP_ERR_FITS = find_segment("Bp_err", BP_ERR_FITS)

print("Field files:")
print(BR_FITS, BT_FITS, BP_FITS, sep="\n")
print("Error files:")
print(BR_ERR_FITS, BT_ERR_FITS, BP_ERR_FITS, sep="\n")

In [ ]:
config = {
    "path": str(RUN_DIR),
    "work_path": str(WORK_DIR),
    "logging": {"project": "nf2", "name": "SHARP CEA"},
    "data": {
        "geometry": "cartesian",
        "boundaries": [{
            "type": "sharp",
            "files": {"Br": str(BR_FITS), "Bt": str(BT_FITS), "Bp": str(BP_FITS)},
            "errors": {"Br_err": str(BR_ERR_FITS), "Bt_err": str(BT_ERR_FITS), "Bp_err": str(BP_ERR_FITS)},
        }],
    },
}

if RUN_TRAINING:
    nf2.run(**config)
else:
    config

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
nf2.export_file(str(NF2_PATH), str(EXPORT_DIR / "field.vtk"), fmt="vtk", Mm_per_pixel=EXPORT_MM_PER_PIXEL, height_range=[HEIGHT_MIN, HEIGHT_MAX], metrics=["j", "alpha", "free_energy"])
nf2.export_file(str(NF2_PATH), str(EXPORT_DIR / "field.npz"), fmt="npz", Mm_per_pixel=EXPORT_MM_PER_PIXEL, height_range=[HEIGHT_MIN, HEIGHT_MAX], metrics=["j", "alpha", "free_energy"])

In [ ]:
out = nf2.load(NF2_PATH)
cube = out.load_cube(
    Mm_per_pixel=EXPORT_MM_PER_PIXEL,
    height_range=[HEIGHT_MIN, HEIGHT_MAX],
    metrics=["j", "free_energy"],
    progress=True,
)

def values(x):
    return np.asarray(getattr(x, "value", x))

b = values(cube["b"])
j = values(cube["metrics"]["j"])
free_energy = values(cube["metrics"]["free_energy"])
metrics = {
    "mean_normalized_divergence": float(np.nanmean(normalized_divergence(b))),
    "theta_J_deg": float(theta_J(b, j)),
    "sigma_J_1e2": float(sigma_J(b, j) * 1e2),
    "mean_B_norm": float(np.nanmean(vector_norm(b))),
    "mean_J_norm": float(np.nanmean(vector_norm(j))),
    "total_free_energy_density": float(np.nansum(free_energy)),
}
metrics

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4))

current_map = np.nansum(vector_norm(j), axis=2)
free_energy_map = np.nansum(free_energy, axis=2)
boundary_bz = b[:, :, 0, 2]

im = axs[0].imshow(current_map.T, origin="lower", cmap="magma")
axs[0].set_title("Integrated |J|")
plt.colorbar(im, ax=axs[0], fraction=0.046)

im = axs[1].imshow(free_energy_map.T, origin="lower", cmap="inferno")
axs[1].set_title("Free energy")
plt.colorbar(im, ax=axs[1], fraction=0.046)

lim = np.nanpercentile(np.abs(boundary_bz), 99)
im = axs[2].imshow(boundary_bz.T, origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim)
axs[2].set_title("Model boundary Bz")
plt.colorbar(im, ax=axs[2], fraction=0.046)

for ax in axs:
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()
plt.show()